# Test cấu trúc chia BINS


In [ ]:
import pandas as pd
import numpy as np

# 1. Create a raw sample dataframe with missing ages
raw_data = pd.DataFrame(
    {"person_age": [22.0, np.nan, 45.0, 29.0, 61.0, 34.0, np.nan, 19.0]}
)

print("--- STEP 1: Raw Data ---")
print(raw_data)
print("-" * 30)

# 2. Simulate Tier 1 Cleaner: Replace NaN with -1.0
clean_data = raw_data.copy()
clean_data["person_age"] = clean_data["person_age"].fillna(-1.0)

print("--- STEP 2: Cleaned Data (Tier 1 Output) ---")
print(clean_data)

--- STEP 1: Raw Data ---
   person_age
0        22.0
1         NaN
2        45.0
3        29.0
4        61.0
5        34.0
6         NaN
7        19.0
------------------------------
--- STEP 2: Cleaned Data (Tier 1 Output) ---
   person_age
0        22.0
1        -1.0
2        45.0
3        29.0
4        61.0
5        34.0
6        -1.0
7        19.0


In [2]:
# Select the column name we want to test
col = "person_age"

# Filter out the -1.0 flags using boolean indexing
valid_data = clean_data[clean_data[col] != -1.0][col]

print("--- STEP 3: Valid Data for Calculation ---")
print(f"Original clean data length: {len(clean_data)}")
print(f"Valid data length (without -1.0): {len(valid_data)}")
print("\nValid data points:")
print(valid_data.values)

--- STEP 3: Valid Data for Calculation ---
Original clean data length: 8
Valid data length (without -1.0): 6

Valid data points:
[22. 45. 29. 61. 34. 19.]


In [3]:
# Set how many bins you want to test (e.g., 3 bins)
n_bins = 3

# Run pd.qcut to get the boundaries (edges)
# We use '_' because we don't need the categorized series yet, only the cut-off points
_, edges = pd.qcut(valid_data, q=n_bins, retbins=True, duplicates="drop")

print("--- STEP 4: Calculated Bin Edges ---")
print(f"Calculated thresholds: {edges}")
print(f"Lowest edge found: {edges[0]}")
print(f"Highest edge found: {edges[-1]}")

--- STEP 4: Calculated Bin Edges ---
Calculated thresholds: [19.         26.66666667 37.66666667 61.        ]
Lowest edge found: 19.0
Highest edge found: 61.0


In [ ]:
print(f"Edges BEFORE insert: {edges}")

# If the lowest boundary is greater than -1.0, insert a new starting wall at -2.0
if edges[0] > -1.0:
    edges = np.insert(edges, 0, -2.0)

print(f"Edges AFTER insert : {edges}")
print("\nWhy did we do this?")
print(
    f"Now, the first bin spans from {edges[0]} to {edges[1]}, which safely contains our -1.0 flag!"
)

Edges BEFORE insert: [19.         26.66666667 37.66666667 61.        ]
Edges AFTER insert : [-2.         19.         26.66666667 37.66666667 61.        ]

Why did we do this?
Now, the first bin spans from -2.0 to 19.0, which safely contains our -1.0 flag!


In [ ]:
# Generate label names based on the final number of edges
group_names = [f"Bin_{i}" for i in range(len(edges) - 1)]
print(f"Generated group names: {group_names}\n")

# Run pd.cut to slice the entire dataset using our modified boundaries
binned_data = pd.cut(
    clean_data[col], bins=edges, include_lowest=True, labels=group_names
).astype(str)

# Combine original clean data and the final binned text side-by-side to verify the logic
verification_df = pd.DataFrame(
    {"Clean_Value": clean_data[col], "Assigned_Bin": binned_data}
)

print("--- FINAL STEP: Verification Table ---")
print(verification_df)

Generated group names: ['Bin_0', 'Bin_1', 'Bin_2', 'Bin_3']

--- FINAL STEP: Verification Table ---
   Clean_Value Assigned_Bin
0         22.0        Bin_1
1         -1.0        Bin_0
2         45.0        Bin_3
3         29.0        Bin_2
4         61.0        Bin_3
5         34.0        Bin_2
6         -1.0        Bin_0
7         19.0        Bin_0


# Test Cấu trúc tính toán WoE


In [ ]:
import pandas as pd
import numpy as np

# 1. Create a controlled testing dataset (10 customers)

# Target: 0 = Good, 1 = Bad

mock_df = pd.DataFrame(
    {
        "home_ownership": [
            "RENT",
            "RENT",
            "RENT",
            "RENT",
            "MORTGAGE",
            "MORTGAGE",
            "MORTGAGE",
            "OWN",
            "OWN",
            "OWN",
        ],
        "loan_status": [0, 1, 1, 1, 0, 0, 1, 0, 0, 0],
    }
)

print("--- Step 1: Mock Dataset for Auditing ---")
print(mock_df)

--- Step 1: Mock Dataset for Auditing ---
  home_ownership  loan_status
0           RENT            0
1           RENT            1
2           RENT            1
3           RENT            1
4       MORTGAGE            0
5       MORTGAGE            0
6       MORTGAGE            1
7            OWN            0
8            OWN            0
9            OWN            0


In [8]:
# 2. Define target and feature columns
X_col = mock_df["home_ownership"]
y_target = mock_df["loan_status"]

# 3. Calculate global totals
total_bad = y_target.sum()
total_good = len(y_target) - total_bad

print("--- Step 2: Global System Statistics ---")
print(f"Total Bad customers (B) : {total_bad}")
print(f"Total Good customers (G): {total_good}\n")

# 4. Perform Pandas aggregation (Groupby)
stats_df = pd.DataFrame({"value": X_col, "target": y_target})
grouped = stats_df.groupby("value")["target"].agg(bad_count="sum", total_count="count")
grouped["good_count"] = grouped["total_count"] - grouped["bad_count"]

# 5. Apply the mathematical formulas with smoothing
grouped["dist_good"] = (grouped["good_count"] + 0.5) / (total_good + 1)
grouped["dist_bad"] = (grouped["bad_count"] + 0.5) / (total_bad + 1)
grouped["woe"] = np.log(grouped["dist_good"] / grouped["dist_bad"])

print("--- Step 3: Intermediate Mathematical Table ---")
print(grouped)

# 6. Convert to final lookup dictionary
woe_lookup = grouped["woe"].to_dict()
print("\n--- Step 4: Final Output Lookup Dictionary ---")
print(woe_lookup)

--- Step 2: Global System Statistics ---
Total Bad customers (B) : 4
Total Good customers (G): 6

--- Step 3: Intermediate Mathematical Table ---
          bad_count  total_count  good_count  dist_good  dist_bad       woe
value                                                                      
MORTGAGE          1            3           2   0.357143       0.3  0.174353
OWN               0            3           3   0.500000       0.1  1.609438
RENT              3            4           1   0.214286       0.7 -1.183770

--- Step 4: Final Output Lookup Dictionary ---
{'MORTGAGE': 0.17435338714477774, 'OWN': 1.6094379124341003, 'RENT': -1.1837700970084166}
